# Transformers and MLflow: Downstream Tasks and Experiment Tracking

1. Install Dependencies

In [1]:
!pip install transformers mlflow datasets torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 918.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

2. Sentiment Classification

In [2]:
import mlflow
import pandas as pd
from transformers import pipeline

# 20 custom sentences with known labels
sentences = [
    ("I absolutely love this product, it works perfectly!", 1),
    ("This is the best movie I have ever seen.", 1),
    ("What a wonderful experience, highly recommend it.", 1),
    ("The food was delicious and the service was amazing.", 1),
    ("I feel so happy and grateful today.", 1),
    ("This book changed my life for the better.", 1),
    ("Excellent quality, arrived on time and well packaged.", 1),
    ("The concert was breathtaking and unforgettable.", 1),
    ("I am so proud of what we accomplished together.", 1),
    ("Great customer support, they resolved my issue instantly.", 1),
    ("This is absolutely terrible, I want a refund.", 0),
    ("Worst experience of my life, never coming back.", 0),
    ("The product broke after one day, complete waste of money.", 0),
    ("I am deeply disappointed with the service.", 0),
    ("The movie was boring and made no sense at all.", 0),
    ("Horrible smell, packaging was damaged and dirty.", 0),
    ("Customer service was rude and unhelpful.", 0),
    ("I regret buying this, it does not work at all.", 0),
    ("The food tasted awful and made me sick.", 0),
    ("Terrible quality, nothing like the description.", 0),
]

texts  = [s[0] for s in sentences]
labels = [s[1] for s in sentences]   # 1=positive, 0=negative

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline("sentiment-analysis", model=MODEL_NAME)

mlflow.set_experiment("Task1_Sentiment_Classification")

with mlflow.start_run(run_name="distilbert_sentiment"):
    results = classifier(texts)

    # Map POSITIVE->1, NEGATIVE->0
    preds = [1 if r["label"] == "POSITIVE" else 0 for r in results]
    accuracy = sum(p == l for p, l in zip(preds, labels)) / len(labels)

    # Log params & metrics
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("num_samples", len(texts))
    mlflow.log_metric("accuracy", accuracy)

    # Save predictions CSV as artifact
    df = pd.DataFrame({
        "sentence": texts,
        "true_label": labels,
        "predicted_label": preds,
        "score": [r["score"] for r in results]
    })
    df.to_csv("sentiment_predictions.csv", index=False)
    mlflow.log_artifact("sentiment_predictions.csv")

    print(f"Accuracy: {accuracy:.4f}")
    print(df.to_string(index=False))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

2026/04/20 16:30:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/20 16:30:40 INFO mlflow.store.db.utils: Updating database tables
2026/04/20 16:30:43 INFO mlflow.tracking.fluent: Experiment with name 'Task1_Sentiment_Classification' does not exist. Creating a new experiment.


Accuracy: 1.0000
                                                 sentence  true_label  predicted_label    score
      I absolutely love this product, it works perfectly!           1                1 0.999880
                 This is the best movie I have ever seen.           1                1 0.999845
        What a wonderful experience, highly recommend it.           1                1 0.999884
      The food was delicious and the service was amazing.           1                1 0.999886
                      I feel so happy and grateful today.           1                1 0.999879
                This book changed my life for the better.           1                1 0.992680
    Excellent quality, arrived on time and well packaged.           1                1 0.999852
          The concert was breathtaking and unforgettable.           1                1 0.999876
          I am so proud of what we accomplished together.           1                1 0.999874
Great customer support,

3. Second Run

In [3]:
MODEL_NAME_2 = "cardiffnlp/twitter-roberta-base-sentiment-latest"
classifier2  = pipeline("sentiment-analysis", model=MODEL_NAME_2)

# This model uses: positive / negative / neutral labels
label_map = {"positive": 1, "negative": 0, "neutral": 0}

with mlflow.start_run(run_name="roberta_sentiment"):
    results2 = classifier2(texts)
    preds2   = [label_map.get(r["label"].lower(), 0) for r in results2]
    accuracy2 = sum(p == l for p, l in zip(preds2, labels)) / len(labels)

    mlflow.log_param("model_name", MODEL_NAME_2)
    mlflow.log_param("num_samples", len(texts))
    mlflow.log_metric("accuracy", accuracy2)

    df2 = pd.DataFrame({
        "sentence": texts,
        "true_label": labels,
        "predicted_label": preds2,
        "score": [r["score"] for r in results2]
    })
    df2.to_csv("sentiment_predictions_roberta.csv", index=False)
    mlflow.log_artifact("sentiment_predictions_roberta.csv")

    print(f"RoBERTa Accuracy: {accuracy2:.4f}")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

RoBERTa Accuracy: 1.0000


4. Named Entity Recognition

In [4]:
from collections import Counter
import json

NER_MODEL = "dslim/bert-base-NER"
ner = pipeline("ner", model=NER_MODEL, aggregation_strategy="simple")

paragraphs = [
    "Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cupertino, California.",
    "Elon Musk is the CEO of Tesla and SpaceX, both headquartered in the United States.",
    "The United Nations headquarters is located in New York City, led by António Guterres.",
    "Barack Obama served as the 44th President of the United States and was born in Hawaii.",
    "Amazon was founded by Jeff Bezos and is based in Seattle, Washington.",
    "The Eiffel Tower is located in Paris, France and was designed by Gustave Eiffel.",
    "Microsoft, founded by Bill Gates and Paul Allen, is headquartered in Redmond, Washington.",
    "Cristiano Ronaldo plays for Al Nassr FC in Riyadh, Saudi Arabia.",
    "The World Health Organization is based in Geneva, Switzerland.",
    "Narendra Modi is the Prime Minister of India, which has its capital in New Delhi.",
]

mlflow.set_experiment("Task2_Named_Entity_Recognition")

with mlflow.start_run(run_name="bert_base_NER_run1"):
    all_entities = []
    output_rows  = []

    for i, para in enumerate(paragraphs):
        entities = ner(para)
        for ent in entities:
            all_entities.append(ent["entity_group"])
            output_rows.append({
                "paragraph_id": i + 1,
                "paragraph":    para[:60] + "...",
                "entity":       ent["word"],
                "entity_type":  ent["entity_group"],
                "score":        round(ent["score"], 4)
            })

    # Count entity types
    counts = Counter(all_entities)
    total  = len(all_entities)

    # Log params & metrics
    mlflow.log_param("model_name",      NER_MODEL)
    mlflow.log_param("num_paragraphs",  len(paragraphs))
    mlflow.log_metric("total_entities", total)
    mlflow.log_metric("PER_count",      counts.get("PER", 0))
    mlflow.log_metric("ORG_count",      counts.get("ORG", 0))
    mlflow.log_metric("LOC_count",      counts.get("LOC", 0))

    # Save artifact
    df_ner = pd.DataFrame(output_rows)
    df_ner.to_csv("ner_output.csv", index=False)
    mlflow.log_artifact("ner_output.csv")

    print(f"Total entities: {total}")
    print(f"Distribution  : {dict(counts)}")
    print(df_ner.to_string(index=False))

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2026/04/20 16:32:06 INFO mlflow.tracking.fluent: Experiment with name 'Task2_Named_Entity_Recognition' does not exist. Creating a new experiment.


Total entities: 45
Distribution  : {'ORG': 8, 'PER': 17, 'LOC': 20}
 paragraph_id                                                       paragraph                    entity entity_type  score
            1 Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cu...                 Apple Inc         ORG 0.9994
            1 Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cu...                Steve Jobs         PER 0.9945
            1 Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cu...             Steve Wozniak         PER 0.9924
            1 Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cu...                 Cupertino         LOC 0.9982
            1 Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cu...                California         LOC 0.9992
            2 Elon Musk is the CEO of Tesla and SpaceX, both headquartered...                        El         PER 0.9414
            2 Elon Musk is the CEO of Tesla and SpaceX, both headquarte

5. Second Run

In [5]:
NER_MODEL_2 = "Jean-Baptiste/roberta-large-ner-english"
ner2 = pipeline("ner", model=NER_MODEL_2, aggregation_strategy="simple")

with mlflow.start_run(run_name="roberta_large_NER_run2"):
    all_entities2 = []
    output_rows2  = []

    for i, para in enumerate(paragraphs):
        entities = ner2(para)
        for ent in entities:
            all_entities2.append(ent["entity_group"])
            output_rows2.append({
                "paragraph_id": i + 1,
                "entity":       ent["word"],
                "entity_type":  ent["entity_group"],
                "score":        round(ent["score"], 4)
            })

    counts2 = Counter(all_entities2)
    total2  = len(all_entities2)

    mlflow.log_param("model_name",      NER_MODEL_2)
    mlflow.log_param("num_paragraphs",  len(paragraphs))
    mlflow.log_metric("total_entities", total2)
    mlflow.log_metric("PER_count",      counts2.get("PER", 0))
    mlflow.log_metric("ORG_count",      counts2.get("ORG", 0))
    mlflow.log_metric("LOC_count",      counts2.get("LOC", 0))

    df_ner2 = pd.DataFrame(output_rows2)
    df_ner2.to_csv("ner_output_roberta.csv", index=False)
    mlflow.log_artifact("ner_output_roberta.csv")

    print(f"RoBERTa NER — Total: {total2}, Distribution: {dict(counts2)}")

config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: Jean-Baptiste/roberta-large-ner-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

vocab.json: 0.00B [00:01, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

RoBERTa NER — Total: 39, Distribution: {'ORG': 9, 'PER': 11, 'LOC': 19}


6. Text Generation

In [6]:
gen = pipeline("text-generation", model="gpt2")

prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a land far away",
    "Scientists have recently discovered that",
    "The best way to learn programming is",
    "In the year 2050, humans will",
]

# Two configurations to compare
configs = [
    {"max_length": 80,  "temperature": 0.7,  "run_name": "gpt2_temp07_len80"},
    {"max_length": 120, "temperature": 1.2,  "run_name": "gpt2_temp12_len120"},
]

mlflow.set_experiment("Task3_Text_Generation")

for cfg in configs:
    with mlflow.start_run(run_name=cfg["run_name"]):
        mlflow.log_param("model_name",   "gpt2")
        mlflow.log_param("temperature",  cfg["temperature"])
        mlflow.log_param("max_length",   cfg["max_length"])
        mlflow.log_param("num_prompts",  len(prompts))

        observations = []
        for i, prompt in enumerate(prompts):
            output = gen(
                prompt,
                max_length=cfg["max_length"],
                temperature=cfg["temperature"],
                do_sample=True,
                pad_token_id=50256
            )
            generated_text = output[0]["generated_text"]

            obs = {
                "prompt_id":      i + 1,
                "prompt":         prompt,
                "generated_text": generated_text,
                "output_length":  len(generated_text.split()),
                "temperature":    cfg["temperature"],
                "max_length":     cfg["max_length"],
            }
            observations.append(obs)
            print(f"\nPrompt {i+1}: {prompt}")
            print(f"Output : {generated_text[:200]}...")

        # Save outputs as artifact
        df_gen = pd.DataFrame(observations)
        fname  = f"generation_{cfg['run_name']}.csv"
        df_gen.to_csv(fname, index=False)
        mlflow.log_artifact(fname)

        # Text observation artifact
        obs_text = f"""
Configuration: temp={cfg['temperature']}, max_length={cfg['max_length']}
-----------------------------------------------------------------------
Observation:
- Lower temperature (0.7) produces more focused, coherent continuations.
- Higher temperature (1.2) produces more creative but less predictable text.
- GPT-2 tends to stay on-topic for factual prompts but drifts on open-ended ones.
- Longer max_length allows more complete thoughts but may introduce repetition.
        """
        with open(f"observations_{cfg['run_name']}.txt", "w") as f:
            f.write(obs_text)
        mlflow.log_artifact(f"observations_{cfg['run_name']}.txt")

        mlflow.log_metric("avg_output_length",
                          sum(o["output_length"] for o in observations) / len(observations))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...


Prompt 2: Once upon a time in a land far away
Output : Once upon a time in a land far away, no human would venture out through it without some form of help from his peers.

Yet all those many hundreds of thousands would be ailing and in dire need of rescu...


Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt 3: Scientists have recently discovered that
Output : Scientists have recently discovered that what we call 'the'modes' can influence our ability to function. Using these brain, neural responses to a lot of different kinds of stressors (and different str...


Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt 4: The best way to learn programming is
Output : The best way to learn programming is to do it without the help you get from reading books.

Let me also say that this could well be due to some bad taste in your mouth.

How To Develop Computer Progra...

Prompt 5: In the year 2050, humans will
Output : In the year 2050, humans will reach the point we've never reached yet," she says. But the researchers are far from done. "Because our brains are different, we have more trouble doing math," Shephey sa...


7. View MLflow UI in Colab

In [11]:
import mlflow

client = mlflow.tracking.MlflowClient()
experiments = client.search_experiments()

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Experiment: {exp.name}")
    print(f"{'='*60}")

    runs = client.search_runs(exp.experiment_id)
    for run in runs:
        print(f"\n  Run: {run.info.run_name}")
        print(f"  Params  : {run.data.params}")
        print(f"  Metrics : {run.data.metrics}")


Experiment: Task3_Text_Generation

  Run: gpt2_temp12_len120
  Params  : {'model_name': 'gpt2', 'temperature': '1.2', 'max_length': '120', 'num_prompts': '5'}
  Metrics : {'avg_output_length': 217.4}

  Run: gpt2_temp07_len80
  Params  : {'model_name': 'gpt2', 'temperature': '0.7', 'max_length': '80', 'num_prompts': '5'}
  Metrics : {'avg_output_length': 185.6}

Experiment: Task2_Named_Entity_Recognition

  Run: roberta_large_NER_run2
  Params  : {'model_name': 'Jean-Baptiste/roberta-large-ner-english', 'num_paragraphs': '10'}
  Metrics : {'total_entities': 39.0, 'PER_count': 11.0, 'ORG_count': 9.0, 'LOC_count': 19.0}

  Run: bert_base_NER_run1
  Params  : {'model_name': 'dslim/bert-base-NER', 'num_paragraphs': '10'}
  Metrics : {'total_entities': 45.0, 'PER_count': 17.0, 'ORG_count': 8.0, 'LOC_count': 20.0}

Experiment: Task1_Sentiment_Classification

  Run: roberta_sentiment
  Params  : {'model_name': 'cardiffnlp/twitter-roberta-base-sentiment-latest', 'num_samples': '20'}
  Metrics